In [ ]:
# ============================================================
# CHUNK 24 — STANDALONE BASIS-FLEXIBILITY VALIDATION
#
# Runs independently in a fresh Google Colab session.
#
# Fixed fitted model: ARMA(2,1) errors.
# Lag-basis dimensions: 3, 5, 7.
# Generating errors: AR(2), ARMA(2,1).
# Known curves: smooth quadratic, narrow peak.
# 200 paired replications per generating scenario.
#
# Checkpoints and results saved directly to Google Drive.
# Rerun this entire chunk to resume successful fits.
# ============================================================

import sys
import subprocess
import importlib.util

# Install only packages that are missing.
packages = {
    "numpy": "numpy",
    "pandas": "pandas",
    "scipy": "scipy",
    "patsy": "patsy",
    "statsmodels": "statsmodels",
    "matplotlib": "matplotlib",
}
missing_packages = [
    package
    for module, package in packages.items()
    if importlib.util.find_spec(module) is None
]
if missing_packages:
    subprocess.check_call([
        sys.executable, "-m", "pip", "install", "-q",
        *missing_packages,
    ])

from pathlib import Path
import hashlib
import json
import platform
import time
import warnings
import zipfile

import numpy as np
import pandas as pd
import scipy
import patsy
import statsmodels
import matplotlib
import matplotlib.pyplot as plt

from numpy.polynomial.legendre import legvander
from scipy.signal import lfilter
from patsy import dmatrix
from statsmodels.tsa.statespace.sarimax import SARIMAX
from IPython.display import display
from google.colab import drive

drive.mount("/content/drive", force_remount=False)

# ----------------------------
# Fixed settings
# ----------------------------

REPLICATES = 200
SEED = 20261007
N_WEEKS = 180
BURN_IN = 500
MAX_LAG = 6
BASIS_DIMENSIONS = (3, 5, 7)
FITTED_ORDER = (2, 0, 1)
EXPOSURE_AR = 0.85

OUTPUT = Path(
    "/content/drive/MyDrive/Paper2_saved_results/"
    "basis_flexibility_standalone_v1"
)
OUTPUT.mkdir(parents=True, exist_ok=True)

CHECKPOINT = OUTPUT / "replication_checkpoint.csv"
SIGNATURE_FILE = OUTPUT / "input_signature.txt"

lag_weeks = np.arange(MAX_LAG + 1)
lag_coordinate = np.linspace(-1.0, 1.0, MAX_LAG + 1)

true_curves = {
    "smooth_curve": (
        0.16
        + 0.06 * lag_coordinate
        - 0.10 * lag_coordinate**2
    ),
    "narrow_peak": (
        0.40
        * np.exp(-0.5 * ((lag_weeks - 2.0) / 0.65)**2)
    ),
}

error_definitions = {
    "AR2_errors": {
        "ar": [1.10, -0.35],
        "ma": [],
        "innovation_sd": 0.25,
    },
    "ARMA21_errors": {
        "ar": [0.90, -0.20],
        "ma": [0.40],
        "innovation_sd": 0.25,
    },
}

# Construct orthonormal polynomial bases.
bases = {}
for dimension in BASIS_DIMENSIONS:
    raw_basis = legvander(lag_coordinate, dimension - 1)
    bases[dimension] = np.linalg.qr(raw_basis)[0]

calendar_time = np.linspace(0.0, 1.0, N_WEEKS)
trend_design = np.asarray(
    dmatrix(
        "bs(t, df=8, degree=3, include_intercept=False)",
        {"t": calendar_time},
        return_type="dataframe",
    ),
    dtype=float,
)

trend_coefficients = np.linspace(
    -0.10, 0.10, trend_design.shape[1]
)
trend_mean = trend_design @ trend_coefficients

versions = {
    "python": platform.python_version(),
    "numpy": np.__version__,
    "pandas": pd.__version__,
    "scipy": scipy.__version__,
    "patsy": patsy.__version__,
    "statsmodels": statsmodels.__version__,
    "matplotlib": matplotlib.__version__,
}

settings = {
    "implementation": "standalone_basis_validation_v1",
    "replicates": REPLICATES,
    "seed": SEED,
    "analyzed_weeks": N_WEEKS,
    "burn_in": BURN_IN,
    "maximum_lag_weeks": MAX_LAG,
    "basis_dimensions": list(BASIS_DIMENSIONS),
    "trend_df": 8,
    "fitted_order": list(FITTED_ORDER),
    "exposure_AR": EXPOSURE_AR,
    "error_definitions": error_definitions,
    "true_curves": {
        key: value.tolist() for key, value in true_curves.items()
    },
    "bases": {
        str(key): value.tolist() for key, value in bases.items()
    },
    "trend_coefficients": trend_coefficients.tolist(),
    "covariance_type": "robust",
    "interval_multiplier": 1.96,
    "optimizer_strategy": (
        "L-BFGS 1500 iterations; if invalid, Powell 2500 "
        "followed by L-BFGS 2500"
    ),
    "pairing": (
        "Identical exposure and errors across basis dimensions "
        "and curve shapes within error scenario and replication"
    ),
    "coverage_target": "Original generating lag curve",
    "versions": versions,
}

signature = hashlib.sha256(
    json.dumps(settings, sort_keys=True).encode()
).hexdigest()

if SIGNATURE_FILE.exists():
    if SIGNATURE_FILE.read_text().strip() != signature:
        raise RuntimeError(
            "Existing results use different settings or package "
            "versions. Restore the original settings/environment "
            "or choose a new OUTPUT folder."
        )
elif CHECKPOINT.exists():
    raise RuntimeError(
        "A checkpoint exists without its input signature. "
        "Choose a new OUTPUT folder."
    )
else:
    SIGNATURE_FILE.write_text(signature, encoding="utf-8")

(OUTPUT / "settings.json").write_text(
    json.dumps(settings, indent=2),
    encoding="utf-8",
)

# ----------------------------
# Standalone fitting function
# ----------------------------

def fit_lag_curve(y, design, basis):
    """Fit fixed ARMA(2,1) regression and propagate robust covariance."""

    model = SARIMAX(
        endog=np.asarray(y, dtype=float),
        exog=np.asarray(design, dtype=float),
        order=FITTED_ORDER,
        trend="n",
        enforce_stationarity=True,
        enforce_invertibility=True,
    )

    dimension = basis.shape[1]
    coefficient_index = np.arange(
        design.shape[1] - dimension,
        design.shape[1],
    )

    attempts = []
    warning_messages = []
    candidates = []
    starter = None

    def attempt(method, maxiter, start_params=None):
        nonlocal starter

        kwargs = {
            "method": method,
            "maxiter": maxiter,
            "disp": False,
            "cov_type": "robust",
        }
        if start_params is not None:
            kwargs["start_params"] = start_params

        try:
            with warnings.catch_warnings(record=True) as caught:
                warnings.simplefilter("always")
                result = model.fit(**kwargs)

            warning_messages.extend(
                str(item.message) for item in caught
                if not issubclass(item.category, DeprecationWarning)
            )

            params = np.asarray(result.params, dtype=float)
            if np.isfinite(params).all():
                starter = params

            converged = bool(
                result.mle_retvals.get("converged", False)
            )
            loglikelihood = float(result.llf)

            record = {
                "optimizer": method,
                "converged": converged,
                "log_likelihood": (
                    loglikelihood
                    if np.isfinite(loglikelihood) else None
                ),
            }

            if not converged or not np.isfinite(loglikelihood):
                record["valid"] = False
                attempts.append(record)
                return

            covariance = np.asarray(
                result.cov_params(), dtype=float
            )
            lag_covariance = covariance[
                np.ix_(coefficient_index, coefficient_index)
            ]
            lag_covariance = (
                lag_covariance + lag_covariance.T
            ) / 2.0

            if not np.isfinite(lag_covariance).all():
                raise ValueError("Nonfinite coefficient covariance.")

            eigenvalues = np.linalg.eigvalsh(lag_covariance)
            tolerance = 1e-8 * max(
                1.0, float(np.max(np.abs(eigenvalues)))
            )

            # Reject materially indefinite covariance.
            if eigenvalues.min() < -tolerance:
                raise ValueError(
                    "Coefficient covariance is materially indefinite."
                )

            # Correct only tiny numerical negative eigenvalues.
            covariance_adjusted = bool(eigenvalues.min() < 0)
            if covariance_adjusted:
                values, vectors = np.linalg.eigh(lag_covariance)
                lag_covariance = (
                    vectors * np.maximum(values, 0.0)
                ) @ vectors.T

            theta = params[coefficient_index]
            curve = basis @ theta
            curve_covariance = basis @ lag_covariance @ basis.T
            variances = np.diag(curve_covariance)

            if (
                not np.isfinite(curve).all()
                or not np.isfinite(variances).all()
                or (variances <= 0).any()
            ):
                raise ValueError("Invalid curve or curve variances.")

            record["valid"] = True
            attempts.append(record)
            candidates.append({
                "curve": curve,
                "se": np.sqrt(variances),
                "llf": loglikelihood,
                "optimizer": method,
                "covariance_adjusted": covariance_adjusted,
            })

        except Exception as error:
            attempts.append({
                "optimizer": method,
                "valid": False,
                "reason": f"{type(error).__name__}: {error}",
            })

    attempt("lbfgs", 1500)

    if not candidates:
        attempt("powell", 2500, starter)
        attempt("lbfgs", 2500, starter)

    if not candidates:
        raise RuntimeError(
            "No converged fit with valid covariance. "
            + json.dumps(attempts)
        )

    selected = max(candidates, key=lambda item: item["llf"])
    selected["attempts"] = json.dumps(attempts)
    selected["warnings"] = " | ".join(
        dict.fromkeys(warning_messages)
    )
    return selected

# ----------------------------
# Checkpoint handling
# ----------------------------

key_columns = [
    "error_scenario",
    "curve_scenario",
    "replicate",
    "basis_dimension",
]

if CHECKPOINT.exists():
    previous = pd.read_csv(CHECKPOINT)
    if not set(key_columns + ["status"]).issubset(previous.columns):
        raise RuntimeError("Unexpected checkpoint schema.")
    if previous.duplicated(key_columns).any():
        raise RuntimeError("Duplicate checkpoint task records.")

    records = {
        tuple(row[column] for column in key_columns): row
        for row in previous.to_dict("records")
    }
else:
    records = {}

def save_checkpoint():
    temporary = CHECKPOINT.with_suffix(".tmp.csv")
    pd.DataFrame(records.values()).to_csv(temporary, index=False)
    temporary.replace(CHECKPOINT)

def stable_rng(error_name, replicate):
    token = f"{SEED}|{error_name}|{replicate}"
    task_seed = int.from_bytes(
        hashlib.sha256(token.encode()).digest()[:8],
        "little",
    )
    return np.random.default_rng(task_seed)

# ----------------------------
# Basis projection audit
# ----------------------------

projection_rows = []

for curve_name, truth in true_curves.items():
    for dimension, basis in bases.items():
        projection = basis @ np.linalg.lstsq(
            basis, truth, rcond=None
        )[0]
        projection_rows.append({
            "curve_scenario": curve_name,
            "basis_dimension": dimension,
            "projection_RMSE": float(
                np.sqrt(np.mean((projection - truth)**2))
            ),
        })

projection_table = pd.DataFrame(projection_rows)
projection_table.to_csv(
    OUTPUT / "basis_projection_error.csv",
    index=False,
)

# ----------------------------
# Paired simulations
# ----------------------------

total_fits = (
    REPLICATES
    * len(error_definitions)
    * len(true_curves)
    * len(bases)
)

existing_successes = sum(
    row.get("status") == "success"
    for row in records.values()
)

print(f"Requested fits: {total_fits:,}")
print(f"Previously successful fits: {existing_successes:,}")
print(f"Saving directly to: {OUTPUT}")

session_start = time.perf_counter()
new_attempts = 0

for error_name, definition in error_definitions.items():
    print(f"\n{error_name}")

    for replicate in range(1, REPLICATES + 1):
        rng = stable_rng(error_name, replicate)

        exposure_length = BURN_IN + N_WEEKS + MAX_LAG
        exposure = lfilter(
            [1.0],
            [1.0, -EXPOSURE_AR],
            rng.normal(size=exposure_length),
        )
        exposure /= np.sqrt(1.0 / (1.0 - EXPOSURE_AR**2))

        indices = BURN_IN + MAX_LAG + np.arange(N_WEEKS)
        lagged_exposure = np.column_stack([
            exposure[indices - lag]
            for lag in lag_weeks
        ])

        errors = lfilter(
            np.r_[1.0, definition["ma"]],
            np.r_[1.0, -np.asarray(definition["ar"])],
            rng.normal(
                scale=definition["innovation_sd"],
                size=BURN_IN + N_WEEKS,
            ),
        )[BURN_IN:]

        for curve_name, truth in true_curves.items():
            y = trend_mean + lagged_exposure @ truth + errors

            for dimension, basis in bases.items():
                key = (
                    error_name,
                    curve_name,
                    replicate,
                    dimension,
                )
                if records.get(key, {}).get("status") == "success":
                    continue

                design = np.column_stack([
                    trend_design,
                    lagged_exposure @ basis,
                ])

                record = {
                    "error_scenario": error_name,
                    "curve_scenario": curve_name,
                    "replicate": replicate,
                    "basis_dimension": dimension,
                    "estimator": "ARMA21",
                    "status": "failed",
                    "reason": "",
                }

                try:
                    fitted = fit_lag_curve(y, design, basis)
                    curve = fitted["curve"]
                    se = fitted["se"]

                    lower = curve - 1.96 * se
                    upper = curve + 1.96 * se
                    covered = (truth >= lower) & (truth <= upper)

                    record.update({
                        "status": "success",
                        "curve_RMSE": float(
                            np.sqrt(np.mean((curve - truth)**2))
                        ),
                        "curve_MAE": float(
                            np.mean(np.abs(curve - truth))
                        ),
                        "pointwise_95_coverage": float(covered.mean()),
                        "mean_interval_width": float(
                            np.mean(upper - lower)
                        ),
                        "selected_optimizer": fitted["optimizer"],
                        "log_likelihood": fitted["llf"],
                        "covariance_adjusted": (
                            fitted["covariance_adjusted"]
                        ),
                        "optimizer_attempts": fitted["attempts"],
                        "warning_messages": fitted["warnings"],
                    })

                    for lag in lag_weeks:
                        record[f"bias_lag_{lag}"] = float(
                            curve[lag] - truth[lag]
                        )
                        record[f"covered_lag_{lag}"] = int(
                            covered[lag]
                        )

                except Exception as error:
                    record["reason"] = (
                        f"{type(error).__name__}: {error}"
                    )

                records[key] = record
                new_attempts += 1
                save_checkpoint()

        if replicate % 10 == 0:
            successes = sum(
                row.get("status") == "success"
                for row in records.values()
            )
            elapsed = time.perf_counter() - session_start
            remaining = total_fits - len(records)
            eta_minutes = (
                elapsed / new_attempts * remaining / 60
                if new_attempts else np.nan
            )
            print(
                f"{replicate}/{REPLICATES} replications; "
                f"{successes:,}/{total_fits:,} successful fits; "
                f"{elapsed / 60:.1f} minutes this session; "
                f"approx. {eta_minutes:.0f} minutes "
                "for unattempted fits"
            )

results = pd.DataFrame(records.values())
results.to_csv(OUTPUT / "replication_results.csv", index=False)

failures = results[results.status != "success"].copy()
failures.to_csv(OUTPUT / "fit_failures.csv", index=False)

# ----------------------------
# Summaries and Monte Carlo SE
# ----------------------------

def mean_mcse(values):
    values = pd.Series(values, dtype=float).dropna()
    count = len(values)
    return (
        float(values.mean()) if count else np.nan,
        float(values.std(ddof=1) / np.sqrt(count))
        if count > 1 else np.nan,
    )

summary_rows = []
lag_rows = []

for keys, group in results.groupby(
    ["error_scenario", "curve_scenario", "basis_dimension"],
    sort=True,
):
    good = group[group.status == "success"]
    identifiers = dict(zip(
        ["error_scenario", "curve_scenario", "basis_dimension"],
        keys,
    ))
    row = {
        **identifiers,
        "attempted_replicates": len(group),
        "successful_replicates": len(good),
        "failed_replicates": len(group) - len(good),
    }

    for metric in [
        "curve_RMSE",
        "curve_MAE",
        "pointwise_95_coverage",
        "mean_interval_width",
    ]:
        mean, mcse = mean_mcse(
            good.get(metric, pd.Series(dtype=float))
        )
        row[f"mean_{metric}"] = mean
        row[f"MCSE_{metric}"] = mcse

    summary_rows.append(row)

    for lag in lag_weeks:
        bias, bias_mcse = mean_mcse(
            good.get(f"bias_lag_{lag}", pd.Series(dtype=float))
        )
        coverage, coverage_mcse = mean_mcse(
            good.get(f"covered_lag_{lag}", pd.Series(dtype=float))
        )
        lag_rows.append({
            **identifiers,
            "lag_weeks": lag,
            "successful_replicates": len(good),
            "mean_bias": bias,
            "bias_MCSE": bias_mcse,
            "pointwise_coverage": coverage,
            "coverage_MCSE": coverage_mcse,
        })

summary = pd.DataFrame(summary_rows)
lag_summary = pd.DataFrame(lag_rows)

summary.to_csv(OUTPUT / "basis_comparison_summary.csv", index=False)
lag_summary.to_csv(OUTPUT / "lag_specific_performance.csv", index=False)

# ----------------------------
# Paired contrasts
# ----------------------------

paired_rows = []

for (error_name, curve_name), group in results.groupby(
    ["error_scenario", "curve_scenario"], sort=True
):
    good = group[group.status == "success"]

    for metric in [
        "curve_RMSE",
        "pointwise_95_coverage",
        "mean_interval_width",
    ]:
        if metric in good.columns and not good.empty:
            wide = good.pivot(
                index="replicate",
                columns="basis_dimension",
                values=metric,
            ).reindex(columns=BASIS_DIMENSIONS)
            complete = wide.dropna()
        else:
            complete = pd.DataFrame(
                columns=BASIS_DIMENSIONS, dtype=float
            )

        for smaller in (3, 5):
            if metric == "curve_RMSE":
                difference = complete[smaller] - complete[7]
                direction = "smaller_basis_minus_basis7"
            else:
                difference = complete[7] - complete[smaller]
                direction = "basis7_minus_smaller_basis"

            mean, mcse = mean_mcse(difference)

            paired_rows.append({
                "error_scenario": error_name,
                "curve_scenario": curve_name,
                "metric": metric,
                "smaller_basis_dimension": smaller,
                "comparison_basis_dimension": 7,
                "difference_direction": direction,
                "complete_paired_replicates": len(complete),
                "mean_paired_difference": mean,
                "difference_MCSE": mcse,
                "MC_lower_95": mean - 1.96 * mcse,
                "MC_upper_95": mean + 1.96 * mcse,
            })

paired = pd.DataFrame(paired_rows)
paired.to_csv(OUTPUT / "paired_basis_contrasts.csv", index=False)

# ----------------------------
# Figures
# ----------------------------

fig, axes = plt.subplots(2, 2, figsize=(10, 7), sharex=True)

for row_index, error_name in enumerate(error_definitions):
    for column_index, curve_name in enumerate(true_curves):
        ax = axes[row_index, column_index]
        subset = summary[
            (summary.error_scenario == error_name)
            & (summary.curve_scenario == curve_name)
        ].sort_values("basis_dimension")

        ax.errorbar(
            subset.basis_dimension,
            subset.mean_curve_RMSE,
            yerr=1.96 * subset.MCSE_curve_RMSE,
            fmt="o-",
            capsize=4,
        )
        ax.set_title(f"{error_name}: {curve_name}")
        ax.set_xticks(BASIS_DIMENSIONS)
        ax.set_ylabel("Mean curve RMSE")
        ax.set_xlabel("Number of lag coefficients")

fig.tight_layout()
fig.savefig(OUTPUT / "basis_RMSE_comparison.png", dpi=300)
fig.savefig(OUTPUT / "basis_RMSE_comparison.pdf")
plt.close(fig)

fig, axes = plt.subplots(2, 2, figsize=(10, 7), sharex=True)

for row_index, error_name in enumerate(error_definitions):
    for column_index, curve_name in enumerate(true_curves):
        ax = axes[row_index, column_index]
        subset = summary[
            (summary.error_scenario == error_name)
            & (summary.curve_scenario == curve_name)
        ].sort_values("basis_dimension")

        ax.errorbar(
            subset.basis_dimension,
            100 * subset.mean_pointwise_95_coverage,
            yerr=196 * subset.MCSE_pointwise_95_coverage,
            fmt="o-",
            capsize=4,
        )
        ax.axhline(95, color="black", linestyle="--")
        ax.set_title(f"{error_name}: {curve_name}")
        ax.set_xticks(BASIS_DIMENSIONS)
        ax.set_ylim(0, 105)
        ax.set_ylabel("Mean pointwise coverage (%)")
        ax.set_xlabel("Number of lag coefficients")

fig.tight_layout()
fig.savefig(OUTPUT / "basis_coverage_comparison.png", dpi=300)
fig.savefig(OUTPUT / "basis_coverage_comparison.pdf")
plt.close(fig)

# ----------------------------
# Interpretation and export
# ----------------------------

(OUTPUT / "README.txt").write_text(
    "PAIRED BASIS-FLEXIBILITY VALIDATION\n\n"
    "Fixed ARMA(2,1) estimator; 3, 5 and 7 lag coefficients.\n"
    "200 replications for each error-process and curve scenario.\n"
    "Identical simulated data used across basis dimensions.\n\n"
    "Coverage targets the original generating curve, not its "
    "projection into a restricted basis. Restricted-basis "
    "coverage therefore includes approximation error.\n\n"
    "Paired contrasts use replications where all three fits "
    "succeeded. Failures are recorded separately.\n"
    "MC intervals quantify simulation uncertainty, not empirical "
    "country-coefficient uncertainty.\n\n"
    "These complete-data experiments do not establish performance "
    "under informative missingness, changing coefficients or "
    "all possible exposure and error processes.\n"
    "The model-selection procedure is not repeated.\n\n"
    "Save this executed Colab notebook separately.\n",
    encoding="utf-8",
)

archive_path = OUTPUT.parent / (
    "Paper2_basis_flexibility_standalone_final.zip"
)

with zipfile.ZipFile(
    archive_path, "w", zipfile.ZIP_DEFLATED
) as archive:
    for path in sorted(OUTPUT.glob("*")):
        if path.is_file():
            archive.write(path, arcname=path.name)

with zipfile.ZipFile(archive_path) as archive:
    damaged = archive.testzip()
    if damaged:
        raise RuntimeError(f"Archive verification failed: {damaged}")

print("\nFINAL SUMMARY")
display(summary)

print("\nPAIRED CONTRASTS")
display(paired)

print("\nBASIS PROJECTION ERROR")
display(projection_table)

if not failures.empty:
    print(
        "\nATTENTION: some fits failed. Summaries are conditional "
        "on success. Rerunning this chunk retries failed fits "
        "and reuses successful fits. Resolve or report failures "
        "before treating comparisons as final."
    )
    display(failures[
        key_columns + ["reason"]
    ])
else:
    print("\nAll 2,400 requested fits succeeded.")

print(f"\nVerified archive saved to Drive:\n{archive_path}")
print("\nSave the notebook: File → Save a copy in Drive.")

# Paper 2: follow-up experiments, Chunks 25–29

Start directly at Chunk 25; the expensive Chunk 24 need not be rerun. All follow-up cells are independent of its runtime variables. The original cell and saved outputs are retained in the updated notebook.

Run 25 then 26 for broader shapes, or 25 then 27 for interval calibration. Run 28–29 to summarize/export either experiment, including a partially completed run. In a fresh session, 25 then 28–29 regenerates reports from saved files without fitting.

The default `FU_MODE = "pilot"` uses 20 outer replications, 99 bootstrap draws per outer replication, and 20 replications per error/shape in the shape comparison. Pilot results must remain labelled pilot. This still requests 240 shape fits and 2,000 bootstrap original/refits.

For the predefined larger experiment, change only `FU_MODE = "full"` in Chunk 25 and rerun 25. Full mode uses 200 replications, 399 bootstrap draws, and requests 2,400 added shape fits and 80,000 bootstrap original/refits. Optimizer retries are additional. Full mode reuses compatible pilot shape fits, fitted original models and bootstrap draws; it extends each bootstrap checkpoint and recalculates intervals. Matching design settings and software versions are required. Keep runtime expectations based on the displayed seconds/refit: this is much more expensive than the original basis comparison.

Only modify settings in Chunk 25 before starting the fitting cells. Settings/environment changes require a new output folder or restoration of the original environment. Each completed shape fit and bootstrap refit is checkpointed directly to Drive. A stopped outer replication resumes using its saved original fit and completed inner draws. `FU_RETRY_FAILED = False` retains failures without silently replacing them; set it to True and rerun 25 and the relevant experiment to retry the same failed tasks with the same random seeds.

The added targets are a broad Gaussian peak centred at three weeks with width 1.5 weeks, and two localized peaks centred at one and five weeks with width 0.65 weeks. They extend the original scenarios; they do not replace them. Exposure/error generation and outer random seeds match Chunk 24. Multiple basis fits share each target-specific y. Shapes share stochastic inputs but have different generating means.

The focused calibration comparison uses the original narrow target, seven basis coefficients and ARMA(2,1)-generated errors. Both intervals use the exact same original simulated y and fitted lag curve. The current interval is the robust-covariance Wald interval. The alternative is a 95% basic parametric bootstrap interval: `[2*estimate - q97.5(bootstrap estimates), 2*estimate - q2.5(bootstrap estimates)]`. Exposure and trend remain fixed; serial errors are regenerated from the fitted Gaussian ARMA(2,1), with exact stationary-state initialization, and every bootstrap sample re-estimates all regression and error parameters. It does not shuffle time-series rows or resample fitted lag coefficients from their covariance matrix.

The bootstrap preserves fitted-model dependence, not arbitrary departures from that model. It does not repeat model selection, evaluate empirical country calendars or establish general robustness to misspecification. It may improve coverage, worsen coverage, or leave it unchanged. Its failure and width results are part of the evidence.

At least 95% of the predeclared bootstrap draws must succeed to construct an interval. Failed draws are recorded, not replaced. Reports include attempted/successful/failed refits, unavailable outer intervals, success-conditional coverage and width, coverage/width on common successful outer datasets, paired differences with Monte Carlo SE, and lag-specific paired coverage. MCSE is calculated between outer replication-level summaries, preserving within-replication dependence across seven lags. These are pointwise intervals, not simultaneous whole-curve bands. Neither inner draws nor the number of model fits is the number of independent outer datasets.

Original results are read only from `/content/drive/MyDrive/Paper2_saved_results/basis_flexibility_standalone_v1`. Matching design settings are checked before they are shown alongside the follow-up results. If package versions differ, original results are historical context rather than a controlled cross-shape comparison. New output folders are `followup_basis_bootstrap_v1_pilot` and `followup_basis_bootstrap_v1_full`, with separate ZIP exports. No original result files are overwritten.

## Sources checked for the implementation
- Statsmodels SARIMAX fitting and robust covariance: https://www.statsmodels.org/devel/generated/statsmodels.tsa.statespace.sarimax.SARIMAX.fit.html
- Statsmodels fitted AR/MA parameter access: https://www.statsmodels.org/dev/generated/statsmodels.tsa.statespace.sarimax.SARIMAXResults.html
- TSA package documentation describes generating time-series bootstrap samples from a fitted ARMA model and refitting: https://stat.ethz.ch/CRAN/web/packages/TSA/refman/TSA.html

## Validation
The chunks were syntax-checked and exercised locally with the actual 180-week fitting design at reduced replication counts. Tests covered exact original exposure/error draws, actual model/bootstrap fits, paired summaries and exports, stationary error moments, completed and interrupted-checkpoint resumption, injected original/refit failures, and expansion of compatible pilot checkpoints. The Google Drive mount was mocked locally; actual Colab/Drive execution and full scientific follow-up experiments remain to be run. No smoke-test results are publication evidence or included in the deliverable results.


## chunk 25 setup

Run first; no Chunk 24 rerun needed.

In [ ]:
# CHUNK 25 — FOLLOW-UP SETUP AND FITTER (no Chunk 24 rerun required)
# Run this cell first in a fresh or existing Colab session.
import sys, subprocess, importlib.util
for module in ("numpy", "pandas", "scipy", "patsy", "statsmodels", "matplotlib"):
    if importlib.util.find_spec(module) is None:
        subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", module])
from pathlib import Path
import hashlib, json, platform, time, warnings, zipfile
import numpy as np
import pandas as pd
import scipy, patsy, statsmodels, matplotlib
import matplotlib.pyplot as plt
from scipy.signal import lfilter
from scipy.linalg import solve_discrete_lyapunov
from numpy.polynomial.legendre import legvander
from patsy import dmatrix
from statsmodels.tsa.statespace.sarimax import SARIMAX
from IPython.display import display
from google.colab import drive

drive.mount("/content/drive", force_remount=False)

# PILOT first. Change to "full" for the predeclared larger experiment.
# Changing mode selects a NEW folder; it does not overwrite pilot/original results.
FU_MODE = "pilot"
if FU_MODE not in ("pilot", "full"):
    raise ValueError("FU_MODE must be 'pilot' or 'full'.")
FU_SHAPE_R = 20 if FU_MODE == "pilot" else 200
FU_BOOT_R = 20 if FU_MODE == "pilot" else 200
FU_BOOT_B = 99 if FU_MODE == "pilot" else 399
# Focused calibration: original narrow target, seven coefficients,
# ARMA(2,1)-generated errors. Optional extra targets/processes cost more fits.
FU_BOOT_SHAPES = ("narrow_peak",)
FU_BOOT_ERRORS = ("ARMA21_errors",)
FU_MIN_BOOT_SUCCESS = 0.95
FU_RETRY_FAILED = False  # failures are retained; toggle True for deterministic retries
FU_SEED = 20261007       # same data generator and outer seeds as Chunk 24
FU_N, FU_BURN, FU_LAG = 180, 500, 6
FU_DIMS = (3, 5, 7)
FU_EXPOSURE_AR = 0.85
FU_ROOT = Path("/content/drive/MyDrive/Paper2_saved_results")
FU_ORIGINAL = FU_ROOT / "basis_flexibility_standalone_v1"
FU_OUT = FU_ROOT / f"followup_basis_bootstrap_v1_{FU_MODE}"
FU_OUT.mkdir(parents=True, exist_ok=True)
FU_LAGS = np.arange(7)
FU_U = np.linspace(-1.0, 1.0, 7)
FU_CURVES = {
    "smooth_curve": 0.16 + 0.06 * FU_U - 0.10 * FU_U**2,
    "narrow_peak": 0.40 * np.exp(-0.5 * ((FU_LAGS - 2.0) / 0.65)**2),
    "broad_peak_followup": 0.40 * np.exp(-0.5 * ((FU_LAGS - 3.0) / 1.5)**2),
    "two_peak_followup": (
        0.30 * np.exp(-0.5 * ((FU_LAGS - 1.0) / 0.65)**2)
        + 0.30 * np.exp(-0.5 * ((FU_LAGS - 5.0) / 0.65)**2)
    ),
}
FU_NEW_SHAPES = ("broad_peak_followup", "two_peak_followup")
FU_ERRORS = {
    "AR2_errors": {"ar": [1.10, -0.35], "ma": [], "innovation_sd": 0.25},
    "ARMA21_errors": {"ar": [0.90, -0.20], "ma": [0.40], "innovation_sd": 0.25},
}
FU_BASES = {k: np.linalg.qr(legvander(FU_U, k-1))[0] for k in FU_DIMS}
FU_TREND = np.asarray(dmatrix(
    "bs(t, df=8, degree=3, include_intercept=False)",
    {"t": np.linspace(0.0, 1.0, FU_N)}, return_type="dataframe"), dtype=float)
FU_TREND_COEF = np.linspace(-0.10, 0.10, FU_TREND.shape[1])
FU_TREND_MEAN = FU_TREND @ FU_TREND_COEF
FU_VERSIONS = {"python": platform.python_version(), "numpy": np.__version__,
    "pandas": pd.__version__, "scipy": scipy.__version__, "patsy": patsy.__version__,
    "statsmodels": statsmodels.__version__, "matplotlib": matplotlib.__version__}
FU_SETTINGS = {
    "implementation": "followup_basis_bootstrap_v1", "mode": FU_MODE,
    "shape_replicates": FU_SHAPE_R, "bootstrap_outer_replicates": FU_BOOT_R,
    "bootstrap_draws": FU_BOOT_B, "bootstrap_shapes": list(FU_BOOT_SHAPES),
    "bootstrap_errors": list(FU_BOOT_ERRORS), "minimum_bootstrap_success": FU_MIN_BOOT_SUCCESS,
    "seed": FU_SEED, "analyzed_weeks": FU_N, "burn_in": FU_BURN,
    "maximum_lag_weeks": FU_LAG, "basis_dimensions": list(FU_DIMS),
    "fitted_order": [2, 0, 1], "exposure_AR": FU_EXPOSURE_AR,
    "error_definitions": FU_ERRORS, "true_curves": {k: v.tolist() for k,v in FU_CURVES.items()},
    "bases": {str(k): v.tolist() for k,v in FU_BASES.items()},
    "trend_coefficients": FU_TREND_COEF.tolist(), "covariance_type": "robust",
    "optimizer": "L-BFGS 1500; invalid fits retry Powell 2500 then L-BFGS 2500",
    "bootstrap": "conditional Gaussian parametric ARMA(2,1), exact stationary error initialization",
    "bootstrap_interval": "basic, pointwise 95%, linear empirical quantiles",
    "bootstrap_refit": "all regression and error parameters; original-fit warm start",
    "pairing": "same outer y for both intervals; same X/errors across shapes and bases",
    "versions": FU_VERSIONS,
}
FU_SIGNATURE = hashlib.sha256(json.dumps(FU_SETTINGS, sort_keys=True).encode()).hexdigest()
fu_signature_path = FU_OUT / "input_signature.txt"
if fu_signature_path.exists():
    if fu_signature_path.read_text().strip() != FU_SIGNATURE:
        raise RuntimeError("Settings/environment differ. Restore them or select a new FU_OUT folder.")
elif any(FU_OUT.iterdir()):
    raise RuntimeError("Nonempty output folder without a signature: select a new folder.")
else:
    fu_signature_path.write_text(FU_SIGNATURE)
(FU_OUT / "settings.json").write_text(json.dumps(FU_SETTINGS, indent=2))

def fu_rng(*tokens):
    token = "|".join(map(str, tokens))
    seed = int.from_bytes(hashlib.sha256(token.encode()).digest()[:8], "little")
    return np.random.default_rng(seed)

def fu_data(error_name, replicate):
    # Identical RNG token and draw order to original Chunk 24.
    rng = fu_rng(FU_SEED, error_name, replicate)
    exposure = lfilter([1.0], [1.0, -FU_EXPOSURE_AR],
        rng.normal(size=FU_BURN + FU_N + FU_LAG))
    exposure /= np.sqrt(1.0 / (1.0 - FU_EXPOSURE_AR**2))
    indices = FU_BURN + FU_LAG + np.arange(FU_N)
    X = np.column_stack([exposure[indices-lag] for lag in FU_LAGS])
    spec = FU_ERRORS[error_name]
    errors = lfilter(np.r_[1.0, spec["ma"]], np.r_[1.0, -np.asarray(spec["ar"])],
        rng.normal(scale=spec["innovation_sd"], size=FU_BURN+FU_N))[FU_BURN:]
    return X, errors

def fu_csv(frame, path):
    temp = path.with_suffix(".tmp.csv")
    frame.to_csv(temp, index=False)
    temp.replace(path)

def fu_npz(path, **arrays):
    temp = path.with_suffix(".tmp.npz")
    np.savez_compressed(temp, **arrays)
    temp.replace(path)

def fu_load_npz(path):
    with np.load(path, allow_pickle=False) as archive:
        return {k: archive[k].copy() for k in archive.files}

def fu_metrics(curve, lower, upper, truth):
    return {"curve_RMSE": float(np.sqrt(np.mean((curve-truth)**2))),
        "pointwise_95_coverage": float(np.mean((truth>=lower) & (truth<=upper))),
        "mean_interval_width": float(np.mean(upper-lower)),
        **{f"covered_lag_{lag}": int(lower[lag]<=truth[lag]<=upper[lag]) for lag in FU_LAGS}}

def fu_fit(y, design, basis, initial_start=None):
    """Fit fixed ARMA(2,1) regression and propagate robust covariance."""

    model = SARIMAX(
        endog=np.asarray(y, dtype=float),
        exog=np.asarray(design, dtype=float),
        order=(2, 0, 1),
        trend="n",
        enforce_stationarity=True,
        enforce_invertibility=True,
    )

    dimension = basis.shape[1]
    coefficient_index = np.arange(
        design.shape[1] - dimension,
        design.shape[1],
    )

    attempts = []
    warning_messages = []
    candidates = []
    starter = None

    def attempt(method, maxiter, start_params=None):
        nonlocal starter

        kwargs = {
            "method": method,
            "maxiter": maxiter,
            "disp": False,
            "cov_type": "robust",
        }
        if start_params is not None:
            kwargs["start_params"] = start_params

        try:
            with warnings.catch_warnings(record=True) as caught:
                warnings.simplefilter("always")
                result = model.fit(**kwargs)

            warning_messages.extend(
                str(item.message) for item in caught
                if not issubclass(item.category, DeprecationWarning)
            )

            params = np.asarray(result.params, dtype=float)
            if np.isfinite(params).all():
                starter = params

            converged = bool(
                result.mle_retvals.get("converged", False)
            )
            loglikelihood = float(result.llf)

            record = {
                "optimizer": method,
                "converged": converged,
                "log_likelihood": (
                    loglikelihood
                    if np.isfinite(loglikelihood) else None
                ),
            }

            if not converged or not np.isfinite(loglikelihood):
                record["valid"] = False
                attempts.append(record)
                return

            covariance = np.asarray(
                result.cov_params(), dtype=float
            )
            lag_covariance = covariance[
                np.ix_(coefficient_index, coefficient_index)
            ]
            lag_covariance = (
                lag_covariance + lag_covariance.T
            ) / 2.0

            if not np.isfinite(lag_covariance).all():
                raise ValueError("Nonfinite coefficient covariance.")

            eigenvalues = np.linalg.eigvalsh(lag_covariance)
            tolerance = 1e-8 * max(
                1.0, float(np.max(np.abs(eigenvalues)))
            )

            # Reject materially indefinite covariance.
            if eigenvalues.min() < -tolerance:
                raise ValueError(
                    "Coefficient covariance is materially indefinite."
                )

            # Correct only tiny numerical negative eigenvalues.
            covariance_adjusted = bool(eigenvalues.min() < 0)
            if covariance_adjusted:
                values, vectors = np.linalg.eigh(lag_covariance)
                lag_covariance = (
                    vectors * np.maximum(values, 0.0)
                ) @ vectors.T

            theta = params[coefficient_index]
            curve = basis @ theta
            curve_covariance = basis @ lag_covariance @ basis.T
            variances = np.diag(curve_covariance)

            if (
                not np.isfinite(curve).all()
                or not np.isfinite(variances).all()
                or (variances <= 0).any()
            ):
                raise ValueError("Invalid curve or curve variances.")

            record["valid"] = True
            attempts.append(record)
            candidates.append({
                "params": params.copy(),
                "ar": np.asarray(result.arparams),
                "ma": np.asarray(result.maparams),
                "sigma2": float(params[list(result.param_names).index("sigma2")]),
                "curve": curve,
                "se": np.sqrt(variances),
                "llf": loglikelihood,
                "optimizer": method,
                "covariance_adjusted": covariance_adjusted,
            })

        except Exception as error:
            attempts.append({
                "optimizer": method,
                "valid": False,
                "reason": f"{type(error).__name__}: {error}",
            })

    attempt("lbfgs", 1500, initial_start)

    if not candidates:
        attempt("powell", 2500, starter)
        attempt("lbfgs", 2500, starter)

    if not candidates:
        raise RuntimeError(
            "No converged fit with valid covariance. "
            + json.dumps(attempts)
        )

    selected = max(candidates, key=lambda item: item["llf"])
    selected["attempts"] = json.dumps(attempts)
    selected["warnings"] = " | ".join(
        dict.fromkeys(warning_messages)
    )
    return selected


# Promotion to full mode reuses compatible pilot fits and saved bootstrap draws.
# Only R/B/mode may differ; all design settings and package versions must match.
fu_pilot = FU_ROOT / "followup_basis_bootstrap_v1_pilot"
if FU_MODE == "full" and (fu_pilot/"settings.json").exists() and not any(
        (FU_OUT/name).exists() for name in ("shape_replications.csv", "bootstrap_checkpoints")):
    import shutil
    pilot_settings = json.loads((fu_pilot/"settings.json").read_text())
    size_fields = {"mode", "shape_replicates", "bootstrap_outer_replicates", "bootstrap_draws"}
    comparable = {k:v for k,v in FU_SETTINGS.items() if k not in size_fields}
    previous = {k:v for k,v in pilot_settings.items() if k not in size_fields}
    if comparable == previous and pilot_settings["bootstrap_draws"] <= FU_BOOT_B:
        source = fu_pilot/"shape_replications.csv"
        if source.exists(): shutil.copy2(source,FU_OUT/source.name)
        source_dir = fu_pilot/"bootstrap_checkpoints"
        if source_dir.exists():
            target_dir = FU_OUT/"bootstrap_checkpoints"
            target_dir.mkdir(exist_ok=True)
            for source in source_dir.glob("*_original.npz"):
                shutil.copy2(source,target_dir/source.name)
            for source in source_dir.glob("*_draws.npz"):
                state = fu_load_npz(source)
                padded = {}
                for k,arr in state.items():
                    shape = (FU_BOOT_B,)+arr.shape[1:]
                    new = np.full(shape,np.nan,dtype=arr.dtype) if k=="curves" else np.zeros(shape,dtype=arr.dtype)
                    new[:len(arr)] = arr
                    padded[k] = new
                fu_npz(target_dir/source.name,**padded)
        (FU_OUT/"pilot_reuse.json").write_text(json.dumps({
            "source":str(fu_pilot),"note":"Matching pilot fits/draws copied; reports recalculated at full R/B."},indent=2))
        print("Compatible pilot checkpoints reused; full mode extends the saved draws.")
    else:
        print("Pilot design/environment differs; full mode will start independently.")

fu_requested_shapes = FU_SHAPE_R * len(FU_ERRORS) * len(FU_NEW_SHAPES) * len(FU_DIMS)
fu_requested_boot = FU_BOOT_R * len(FU_BOOT_ERRORS) * len(FU_BOOT_SHAPES) * (1+FU_BOOT_B)
print(f"Mode: {FU_MODE}. Shape fits: {fu_requested_shapes:,}; bootstrap original/refits: {fu_requested_boot:,}.")
print("Each fit may require optimizer retries. Full mode is substantially more expensive.")
print(f"Saving/resuming: {FU_OUT}")
print("No need to execute Chunk 24 again. Run 26 for shapes; 27 for bootstrap; 28–29 for reports.")


## chunk 26 shape followup

Broader target shapes; checkpoints after each fit.

In [ ]:
# CHUNK 26 — BROAD-PEAK AND TWO-PEAK FOLLOW-UP (run after 25)
# Original results are not modified. Checkpoint after every requested fit.
fu_shape_path = FU_OUT / "shape_replications.csv"
fu_shape_keys = ["error_scenario", "curve_scenario", "replicate", "basis_dimension"]
fu_shape_records = {}
if fu_shape_path.exists():
    fu_old = pd.read_csv(fu_shape_path)
    if fu_old.duplicated(fu_shape_keys).any():
        raise RuntimeError("Duplicate shape checkpoint keys.")
    fu_shape_records = {tuple(row[k] for k in fu_shape_keys): row
        for row in fu_old.to_dict("records")}
fu_projection = pd.DataFrame([
    {"curve_scenario": name, "basis_dimension": k,
     "projection_RMSE": float(np.sqrt(np.mean((basis @ (basis.T @ truth)-truth)**2)))}
    for name,truth in FU_CURVES.items() for k,basis in FU_BASES.items()])
fu_csv(fu_projection, FU_OUT / "projection_audit.csv")
fu_shape_start, fu_shape_new = time.perf_counter(), 0
for error_name in FU_ERRORS:
    for replicate in range(1, FU_SHAPE_R+1):
        X, errors = fu_data(error_name, replicate)
        for curve_name in FU_NEW_SHAPES:
            truth = FU_CURVES[curve_name]
            y = FU_TREND_MEAN + X @ truth + errors
            for dimension, basis in FU_BASES.items():
                key = (error_name, curve_name, replicate, dimension)
                old = fu_shape_records.get(key)
                if old and (old["status"] == "success" or not FU_RETRY_FAILED):
                    continue
                design = np.column_stack([FU_TREND, X @ basis])
                row = dict(zip(fu_shape_keys, key))
                row.update(experiment="followup_shapes", status="failed", reason="")
                started = time.perf_counter()
                try:
                    f = fu_fit(y, design, basis)
                    row.update(status="success", **fu_metrics(
                        f["curve"], f["curve"]-1.96*f["se"], f["curve"]+1.96*f["se"], truth),
                        optimizer_attempts=f["attempts"], warning_messages=f["warnings"],
                        covariance_adjusted=f["covariance_adjusted"], log_likelihood=f["llf"])
                    for lag in FU_LAGS:
                        row[f"estimate_lag_{lag}"] = float(f["curve"][lag])
                        row[f"se_lag_{lag}"] = float(f["se"][lag])
                except Exception as exc:
                    row["reason"] = f"{type(exc).__name__}: {exc}"
                row["fit_seconds"] = time.perf_counter()-started
                fu_shape_records[key] = row
                fu_csv(pd.DataFrame(fu_shape_records.values()), fu_shape_path)
                fu_shape_new += 1
                if fu_shape_new % 10 == 0:
                    remaining = fu_requested_shapes-len(fu_shape_records)
                    rate = (time.perf_counter()-fu_shape_start)/fu_shape_new
                    print(f"Shapes: {len(fu_shape_records)}/{fu_requested_shapes}; "
                          f"approximately {rate*remaining/60:.1f} min for unattempted fits", flush=True)
fu_shape_results = pd.DataFrame(fu_shape_records.values())
fu_csv(fu_shape_results.loc[fu_shape_results.status != "success"], FU_OUT / "shape_failures.csv")
print("Shape follow-up finished. Failures are retained; set FU_RETRY_FAILED=True to retry them.")
print("Pilot mode is not a final coverage experiment. Run 28–29 to summarize/export.")


## chunk 27 bootstrap

Focused serial-dependence-preserving bootstrap; expensive.

In [ ]:
# CHUNK 27 — PAIRED WALD VS ARMA PARAMETRIC BOOTSTRAP (after 25)
# Same original simulated y and same point estimate for both interval methods.
# Fixed seven-coefficient model; no basis/model selection inside the bootstrap.
# Every inner draw is saved; interrupted runs resume at the next unfinished draw.
# Failure draws are retained, not replaced with fresh random draws.

def fu_stationary_errors(ar, ma, sigma2, n, rng):
    # State [e_(t-1), e_(t-2), innovation_(t-1)] before generating e_t.
    if len(ar) != 2 or len(ma) != 1 or not np.isfinite(sigma2) or sigma2 <= 0:
        raise ValueError("Expected finite ARMA(2,1) parameters and positive sigma2.")
    transition = np.array([[ar[0], ar[1], ma[0]], [1., 0., 0.], [0., 0., 0.]])
    if max(abs(np.linalg.eigvals(transition))) >= 1:
        raise ValueError("Bootstrap generator is not stationary.")
    loading = np.array([1., 0., 1.])
    cov = solve_discrete_lyapunov(transition, sigma2*np.outer(loading, loading))
    cov = (cov+cov.T)/2
    values,vectors = np.linalg.eigh(cov)
    if values.min() < -1e-8*max(1., max(abs(values))):
        raise ValueError("Invalid stationary state covariance.")
    state = vectors @ (np.sqrt(np.maximum(values, 0.))*rng.normal(size=3))
    innovations = rng.normal(scale=np.sqrt(sigma2), size=n)
    errors = np.empty(n)
    for i,a in enumerate(innovations):
        state = transition @ state + loading*a
        errors[i] = state[0]
    return errors

fu_boot_dir = FU_OUT / "bootstrap_checkpoints"
fu_boot_dir.mkdir(exist_ok=True)
fu_boot_index_path = FU_OUT / "bootstrap_outer_records.csv"
fu_boot_keys = ["error_scenario", "curve_scenario", "replicate"]
fu_boot_records = {}
if fu_boot_index_path.exists():
    old = pd.read_csv(fu_boot_index_path)
    if old.duplicated(fu_boot_keys).any():
        raise RuntimeError("Duplicate bootstrap outer keys.")
    fu_boot_records = {tuple(row[k] for k in fu_boot_keys): row for row in old.to_dict("records")}
fu_boot_session, fu_boot_new, fu_boot_new_seconds = time.perf_counter(), 0, 0.
for error_name in FU_BOOT_ERRORS:
    for curve_name in FU_BOOT_SHAPES:
        for replicate in range(1, FU_BOOT_R+1):
            key = (error_name, curve_name, replicate)
            tag = f"{error_name}_{curve_name}_{replicate:04d}"
            original_path = fu_boot_dir / f"{tag}_original.npz"
            checkpoint_path = fu_boot_dir / f"{tag}_draws.npz"
            original_failure_path = fu_boot_dir / f"{tag}_original_failure.json"
            X, errors = fu_data(error_name, replicate)
            truth, basis = FU_CURVES[curve_name], FU_BASES[7]
            design = np.column_stack([FU_TREND, X @ basis])
            y = FU_TREND_MEAN + X @ truth + errors
            row = dict(zip(fu_boot_keys, key))
            row.update(experiment="followup_interval_calibration", basis_dimension=7,
                original_status="failed", bootstrap_status="unavailable", reason="",
                bootstrap_requested=FU_BOOT_B, bootstrap_attempted=0,
                bootstrap_successful=0, bootstrap_failed=0)
            if original_failure_path.exists() and not FU_RETRY_FAILED and not original_path.exists():
                fu_boot_records[key] = json.loads(original_failure_path.read_text())
                fu_csv(pd.DataFrame(fu_boot_records.values()), fu_boot_index_path)
                continue
            try:
                if original_path.exists():
                    f = fu_load_npz(original_path)
                else:
                    t0 = time.perf_counter()
                    fitted = fu_fit(y, design, basis)
                    f = {k: np.asarray(fitted[k]) for k in ("curve", "se", "params", "ar", "ma", "sigma2")}
                    f["fit_seconds"] = np.asarray(time.perf_counter()-t0)
                    f["attempts"] = np.asarray(fitted["attempts"])
                    f["warnings"] = np.asarray(fitted["warnings"])
                    fu_npz(original_path, **f)
                row["original_status"] = "success"
                wald = fu_metrics(f["curve"], f["curve"]-1.96*f["se"], f["curve"]+1.96*f["se"], truth)
                row.update({"wald_"+k:v for k,v in wald.items()})
                row["original_fit_seconds"] = float(f["fit_seconds"])
                if checkpoint_path.exists():
                    state = fu_load_npz(checkpoint_path)
                    if state["done"].shape != (FU_BOOT_B,):
                        raise RuntimeError("Bootstrap checkpoint shape differs from settings.")
                else:
                    state = {"done": np.zeros(FU_BOOT_B, dtype=bool),
                        "success": np.zeros(FU_BOOT_B, dtype=bool),
                        "curves": np.full((FU_BOOT_B, 7), np.nan),
                        "seconds": np.zeros(FU_BOOT_B),
                        "reasons": np.full(FU_BOOT_B, "", dtype="U2000"),
                        "attempts": np.full(FU_BOOT_B, "", dtype="U8000"),
                        "warnings": np.full(FU_BOOT_B, "", dtype="U8000")}
                # Coefficients occupy the exogenous-design positions in SARIMAX with trend='n'.
                fitted_mean = design @ f["params"][:design.shape[1]]
                for b in range(FU_BOOT_B):
                    if state["done"][b] and (state["success"][b] or not FU_RETRY_FAILED):
                        continue
                    t0 = time.perf_counter()
                    rng = fu_rng(FU_SEED, "parametric_bootstrap", error_name, curve_name, replicate, b+1)
                    try:
                        bootstrap_y = fitted_mean + fu_stationary_errors(
                            f["ar"], f["ma"], float(f["sigma2"]), FU_N, rng)
                        fb = fu_fit(bootstrap_y, design, basis, initial_start=f["params"])
                        state["curves"][b] = fb["curve"]
                        state["success"][b] = True
                        state["reasons"][b] = ""
                        state["attempts"][b] = fb["attempts"]
                        state["warnings"][b] = fb["warnings"]
                    except Exception as exc:
                        state["success"][b] = False
                        state["curves"][b] = np.nan
                        state["reasons"][b] = f"{type(exc).__name__}: {exc}"
                    state["done"][b] = True
                    state["seconds"][b] = time.perf_counter()-t0
                    fu_npz(checkpoint_path, **state)
                    fu_boot_new += 1
                    fu_boot_new_seconds += state["seconds"][b]
                    if fu_boot_new % 10 == 0:
                        print(f"{tag}: {int(state['done'].sum())}/{FU_BOOT_B} bootstrap draws saved; "
                              f"{fu_boot_new_seconds/fu_boot_new:.1f} sec/refit this session", flush=True)
                good = state["done"] & state["success"]
                row.update(bootstrap_attempted=int(state["done"].sum()),
                    bootstrap_successful=int(good.sum()),
                    bootstrap_failed=int((state["done"] & ~state["success"]).sum()))
                if state["done"].all() and good.sum() >= max(20, int(np.ceil(FU_MIN_BOOT_SUCCESS*FU_BOOT_B))):
                    qlow,qhigh = np.quantile(state["curves"][good], [.025,.975], axis=0, method="linear")
                    lower,upper = 2*f["curve"]-qhigh, 2*f["curve"]-qlow
                    bm = fu_metrics(f["curve"], lower, upper, truth)
                    row.update(bootstrap_status="success", **{"bootstrap_"+k:v for k,v in bm.items()})
                    fu_npz(fu_boot_dir/f"{tag}_intervals.npz", truth=truth, curve=f["curve"],
                        wald_lower=f["curve"]-1.96*f["se"], wald_upper=f["curve"]+1.96*f["se"],
                        bootstrap_lower=lower, bootstrap_upper=upper)
                else:
                    row["bootstrap_status"] = "failed"
                    row["reason"] = "Insufficient successful bootstrap refits under predeclared 95% rule."
            except Exception as exc:
                row["reason"] = f"{type(exc).__name__}: {exc}"
                if row["original_status"] != "success":
                    original_failure_path.write_text(json.dumps(row, indent=2))
            fu_boot_records[key] = row
            fu_csv(pd.DataFrame(fu_boot_records.values()), fu_boot_index_path)
            print(f"Outer {tag}: {row['bootstrap_status']}; "
                  f"{row['bootstrap_successful']}/{FU_BOOT_B} successful refits", flush=True)
print("Bootstrap experiment finished. Run 28–29. Pilot outputs must remain labelled pilot.")


## chunk 28 summaries

Load saved results and report comparisons and failures.

In [ ]:
# CHUNK 28 — SUMMARIES, PAIRED CONTRASTS AND FAILURE ACCOUNTING
# Can run after 25 in a fresh session: loads saved results; no fitting.
def fu_mcse(values):
    a = np.asarray(pd.Series(values).dropna(), dtype=float)
    return (float(a.mean()) if len(a) else np.nan,
            float(a.std(ddof=1)/np.sqrt(len(a))) if len(a)>1 else np.nan)

fu_frames = []
fu_shape_path = FU_OUT / "shape_replications.csv"
if fu_shape_path.exists():
    followup = pd.read_csv(fu_shape_path)
    fu_frames.append(followup)
# Read original results only if design/settings match; never write to original folder.
fu_original_file = FU_ORIGINAL / "replication_results.csv"
if fu_original_file.exists() and (FU_ORIGINAL/"settings.json").exists():
    original_settings = json.loads((FU_ORIGINAL/"settings.json").read_text())
    for field,expected in [("seed",FU_SEED),("analyzed_weeks",FU_N),
            ("burn_in",FU_BURN),("maximum_lag_weeks",FU_LAG),
            ("exposure_AR",FU_EXPOSURE_AR),("covariance_type","robust")]:
        if original_settings.get(field) != expected:
            raise RuntimeError(f"Original design differs: {field}. Do not combine these experiments.")
    if original_settings.get("error_definitions") != FU_ERRORS:
        raise RuntimeError("Original generating errors differ.")
    if original_settings.get("fitted_order") != [2,0,1]:
        raise RuntimeError("Original fitted error model differs.")
    for k,basis in FU_BASES.items():
        if not np.allclose(np.asarray(original_settings["bases"][str(k)]),basis):
            raise RuntimeError("Original basis differs.")
    if not np.allclose(original_settings["trend_coefficients"],FU_TREND_COEF):
        raise RuntimeError("Original trend differs.")
    for name in ("smooth_curve","narrow_peak"):
        if not np.allclose(original_settings["true_curves"][name],FU_CURVES[name]):
            raise RuntimeError("Original target differs.")
    if original_settings.get("versions") != FU_VERSIONS:
        print("Original package versions differ: show original results as historical context, "
              "not as a controlled cross-shape comparison with the follow-up fits.")
    original = pd.read_csv(fu_original_file)
    original["experiment"] = "original_chunk24"
    fu_frames.append(original)
    (FU_OUT/"original_results_provenance.json").write_text(json.dumps({
        "file":str(fu_original_file),
        "sha256":hashlib.sha256(fu_original_file.read_bytes()).hexdigest(),
        "same_package_versions": original_settings.get("versions") == FU_VERSIONS,
        "note":"Original outcomes retained; no rerun. Cross-shape Monte Carlo contrasts are not pooled."
    },indent=2))
else:
    print("Original saved results not found. New shape summaries will still be produced.")

fu_summary_rows, fu_contrast_rows = [], []
if fu_frames:
    combined = pd.concat(fu_frames, ignore_index=True)
    group_cols = ["experiment","error_scenario","curve_scenario","basis_dimension"]
    for key,g in combined.groupby(group_cols,sort=True):
        good = g[g.status=="success"]
        row = dict(zip(group_cols,key))
        row.update(attempted=len(g), successful=len(good), failed=len(g)-len(good))
        for metric in ("curve_RMSE","pointwise_95_coverage","mean_interval_width"):
            mean,se = fu_mcse(good.get(metric,pd.Series(dtype=float)))
            row["mean_"+metric],row["MCSE_"+metric] = mean,se
        fu_summary_rows.append(row)
    fu_basis_summary = pd.DataFrame(fu_summary_rows)
    fu_csv(fu_basis_summary,FU_OUT/"basis_summary_with_original.csv")
    # Paired basis contrasts use only replications where all three fits succeeded.
    for key,g in combined.groupby(["experiment","error_scenario","curve_scenario"]):
        good = g[g.status=="success"]
        for metric in ("curve_RMSE","pointwise_95_coverage","mean_interval_width"):
            pivot = (good.pivot(index="replicate",columns="basis_dimension",values=metric)
                     if metric in good.columns else pd.DataFrame(columns=FU_DIMS))
            complete = pivot.reindex(columns=FU_DIMS).dropna()
            for small in (3,5):
                mean,se = fu_mcse(complete[7]-complete[small])
                fu_contrast_rows.append({"experiment":key[0],"error_scenario":key[1],
                    "curve_scenario":key[2],"metric":metric,"contrast":f"7 minus {small}",
                    "paired_replicates":len(complete),"mean_difference":mean,"MCSE":se,
                    "MC_lower":mean-1.96*se,"MC_upper":mean+1.96*se})
    fu_basis_contrasts = pd.DataFrame(fu_contrast_rows)
    fu_csv(fu_basis_contrasts,FU_OUT/"paired_basis_contrasts_followup.csv")
    display(fu_basis_summary)
    display(fu_basis_contrasts)

fu_boot_summary_rows, fu_boot_contrast_rows, fu_failure_rows = [], [], []
fu_outer_path = FU_OUT/"bootstrap_outer_records.csv"
if fu_outer_path.exists():
    outer = pd.read_csv(fu_outer_path)
    for key,g in outer.groupby(["error_scenario","curve_scenario"]):
        paired = g[(g.original_status=="success") & (g.bootstrap_status=="success")]
        for method,status in (("wald","original_status"),("bootstrap","bootstrap_status")):
            eligible = g[g[status]=="success"]
            row = {"error_scenario":key[0],"curve_scenario":key[1],"interval_method":method,
                "requested_outer_replicates":FU_BOOT_R,"outer_records":len(g),
                "unrecorded_outer_replicates":FU_BOOT_R-len(g),
                "available_intervals":len(eligible),"unavailable_intervals":len(g)-len(eligible),
                "paired_available_intervals":len(paired),
                "bootstrap_draws_attempted":int(g.bootstrap_attempted.sum()),
                "bootstrap_draws_successful":int(g.bootstrap_successful.sum()),
                "bootstrap_draws_failed":int(g.bootstrap_failed.sum())}
            for metric in ("pointwise_95_coverage","mean_interval_width"):
                values = eligible.get(method+"_"+metric,pd.Series(dtype=float))
                mean,se = fu_mcse(values)
                row["success_conditional_mean_"+metric] = mean
                row["success_conditional_MCSE_"+metric] = se
                mean,se = fu_mcse(paired.get(method+"_"+metric,pd.Series(dtype=float)))
                row["paired_mean_"+metric],row["paired_MCSE_"+metric] = mean,se
            fu_boot_summary_rows.append(row)
        for metric in ("pointwise_95_coverage","mean_interval_width"):
            values = (paired.get("bootstrap_"+metric,pd.Series(dtype=float))
                      - paired.get("wald_"+metric,pd.Series(dtype=float)))
            mean,se = fu_mcse(values)
            fu_boot_contrast_rows.append({"error_scenario":key[0],"curve_scenario":key[1],
                "metric":metric,"contrast":"bootstrap minus Wald", "paired_replicates":len(paired),
                "mean_difference":mean,"MCSE":se,"MC_lower":mean-1.96*se,"MC_upper":mean+1.96*se})
    lag_rows = []
    for key,g in outer.groupby(["error_scenario","curve_scenario"]):
        paired = g[(g.original_status=="success") & (g.bootstrap_status=="success")]
        for method in ("wald","bootstrap"):
            for lag in FU_LAGS:
                mean,se = fu_mcse(paired.get(f"{method}_covered_lag_{lag}",pd.Series(dtype=float)))
                lag_rows.append({"error_scenario":key[0],"curve_scenario":key[1],
                    "interval_method":method,"lag_weeks":lag,"paired_replicates":len(paired),
                    "pointwise_coverage":mean,"coverage_MCSE":se})
    fu_csv(pd.DataFrame(lag_rows),FU_OUT/"paired_lag_specific_coverage.csv")
    fu_boot_summary = pd.DataFrame(fu_boot_summary_rows)
    fu_csv(fu_boot_summary,FU_OUT/"interval_calibration_summary.csv")
    fu_csv(pd.DataFrame(fu_boot_contrast_rows),FU_OUT/"paired_interval_contrasts.csv")
    fu_csv(outer[(outer.original_status!="success") | (outer.bootstrap_status!="success")],
           FU_OUT/"outer_interval_failures.csv")
    for path in sorted((FU_OUT/"bootstrap_checkpoints").glob("*_draws.npz")):
        state = fu_load_npz(path)
        for index in np.flatnonzero(state["done"] & ~state["success"]):
            fu_failure_rows.append({"checkpoint":path.name,"bootstrap_draw":int(index+1),
                "reason":str(state["reasons"][index]),"seconds":float(state["seconds"][index])})
    fu_csv(pd.DataFrame(fu_failure_rows,columns=["checkpoint","bootstrap_draw","reason","seconds"]),
           FU_OUT/"bootstrap_refit_failures.csv")
    display(fu_boot_summary)
    display(pd.DataFrame(fu_boot_contrast_rows))
else:
    print("No completed outer bootstrap records yet. Rerun 27 to resume.")
if FU_MODE == "pilot":
    print("PILOT ONLY: small outer R and coarse inner quantiles; do not report final calibration.")
print("Coverage is mean pointwise coverage across seven lags, not whole-curve coverage.")
print("MCSE uses between-outer-replication variation, retaining dependence across lags.")
print("Paired contrasts condition on both methods being available; report failures alongside them.")
print("Bootstrap evaluates the fixed fitted ARMA model, not empirical country calendars or model selection.")


## chunk 29 figures export

Figures and verified Google Drive archive.

In [ ]:
# CHUNK 29 — FIGURES AND VERIFIED DRIVE ARCHIVE (after 28)
# Reads saved tables only; no model fits.
fu_summary_path = FU_OUT/"basis_summary_with_original.csv"
if fu_summary_path.exists():
    tab = pd.read_csv(fu_summary_path)
    names = [s for s in FU_CURVES if s in set(tab.curve_scenario)]
    fig,axes = plt.subplots(3,len(names),figsize=(4*len(names),9),squeeze=False)
    metrics = ["curve_RMSE","pointwise_95_coverage","mean_interval_width"]
    for j,name in enumerate(names):
        for i,metric in enumerate(metrics):
            for error_name in FU_ERRORS:
                part = tab[(tab.curve_scenario==name)&(tab.error_scenario==error_name)].sort_values("basis_dimension")
                part = part[np.isfinite(part["mean_"+metric])]
                if part.empty: continue
                scale = 100 if metric=="pointwise_95_coverage" else 1
                axes[i,j].errorbar(part.basis_dimension,scale*part["mean_"+metric],
                    yerr=1.96*scale*part["MCSE_"+metric],fmt="o-",capsize=3,label=error_name)
            axes[i,j].set_xticks(FU_DIMS)
            axes[i,j].set_ylabel({"curve_RMSE":"Curve RMSE","pointwise_95_coverage":"Pointwise coverage (%)",
                                 "mean_interval_width":"Mean interval width"}[metric])
            if i==1: axes[i,j].axhline(95,color="black",ls="--",lw=1)
            if i==0: axes[i,j].set_title(name.replace("_"," "))
            if i==2: axes[i,j].set_xlabel("Lag-basis coefficients")
    axes[0,0].legend(fontsize=8)
    fig.suptitle(f"Original and separately labelled follow-up shapes — {FU_MODE}")
    fig.tight_layout()
    for ext in ("png","pdf"):fig.savefig(FU_OUT/f"followup_basis_comparison.{ext}",dpi=300,bbox_inches="tight")
    plt.show();plt.close(fig)
fu_interval_path = FU_OUT/"interval_calibration_summary.csv"
if fu_interval_path.exists():
    tab = pd.read_csv(fu_interval_path)
    fig,axes = plt.subplots(1,2,figsize=(10,4))
    labels = [f"{r.curve_scenario}: {r.interval_method}" for r in tab.itertuples()]
    for ax,metric,scale in zip(axes,["pointwise_95_coverage","mean_interval_width"],[100,1]):
        valid = np.isfinite(tab["paired_mean_"+metric])
        if valid.any():
            ax.errorbar(np.arange(len(tab))[valid],scale*tab.loc[valid,"paired_mean_"+metric],
                yerr=1.96*scale*tab.loc[valid,"paired_MCSE_"+metric],fmt="o",capsize=4)
        ax.set_xticks(np.arange(len(tab)),labels,rotation=30,ha="right")
        ax.set_ylabel("Paired pointwise coverage (%)" if scale==100 else "Paired mean interval width")
    axes[0].axhline(95,color="black",ls="--")
    fig.suptitle(f"Fixed seven-coefficient ARMA bootstrap comparison — {FU_MODE}")
    fig.tight_layout()
    for ext in ("png","pdf"):fig.savefig(FU_OUT/f"interval_calibration_comparison.{ext}",dpi=300,bbox_inches="tight")
    plt.show();plt.close(fig)
(FU_OUT/"README_followup.txt").write_text(
    "FOLLOW-UP EXPERIMENTS; original Chunk 24 results retained separately.\n"
    f"Mode={FU_MODE}; shape R={FU_SHAPE_R}; bootstrap outer R={FU_BOOT_R}; inner B={FU_BOOT_B}.\n"
    "Broad peak and two peaks are added targets, not replacements for original targets.\n"
    "Exposure/error draws use the original seed token/draw order; fits are paired within each target.\n"
    "The interval comparison uses the same original y/point estimate for robust Wald and basic bootstrap.\n"
    "Gaussian bootstrap errors follow fitted ARMA(2,1) dependence with exact stationary initialization.\n"
    "Exposure/trend designs remain fixed; all regression/error parameters are re-estimated per draw.\n"
    "Bootstrap failures are retained. At least 95% of requested draws must succeed for an interval.\n"
    "Coverage is conditional on successful interval construction; paired summaries use common outer records.\n"
    "Failure counts, Monte Carlo SE, and widths must accompany coverage. Figures use 1.96-MCSE bars.\n"
    "Paired fits/inner draws are not independent outer datasets. No model selection is repeated.\n"
    "This does not validate empirical calendar timing, missingness or arbitrary error misspecification.\n"
    "Pilot results are not final calibration evidence. Full mode is a computationally expensive follow-up.\n"
    "Original exposure seeds are preserved; bootstrap seeds are independently hashed per outer/draw task.\n")
fu_files = [p for p in FU_OUT.rglob("*") if p.is_file() and ".tmp." not in p.name]
fu_manifest = [{"file":str(p.relative_to(FU_OUT)),"bytes":p.stat().st_size,
                "sha256":hashlib.sha256(p.read_bytes()).hexdigest()} for p in fu_files
               if p.name!="manifest.json"]
(FU_OUT/"manifest.json").write_text(json.dumps(fu_manifest,indent=2))
fu_archive = FU_OUT.parent/f"Paper2_followup_basis_bootstrap_{FU_MODE}.zip"
with zipfile.ZipFile(fu_archive,"w",zipfile.ZIP_DEFLATED) as archive:
    for p in sorted(FU_OUT.rglob("*")):
        if p.is_file() and ".tmp." not in p.name:archive.write(p,p.relative_to(FU_OUT))
with zipfile.ZipFile(fu_archive) as archive:
    if archive.testzip() is not None:raise RuntimeError("Archive integrity check failed.")
print(f"Verified archive saved directly to Drive: {fu_archive}")
print("Use File → Save a copy in Drive to preserve this executed notebook too.")
